# Grouped Query Paged Attention (GQPA)
*Combining GQA's KV-head sharing with a paged block table to serve dynamic-length sequences without memory fragmentation.*

---
**Component:** `learning/leetgpu/gqpa.ipynb`
**Builds on:** `grouped_query_attention.ipynb`, `naive_gemm.py`, `softmax_attention_cutedsl.py`


## Abstract

*A library with a fixed number of numbered shelves and no way to move books once placed wastes a lot of shelf space — short books take up full shelves, and once a sequence ends, the shelf sits empty until manually cleared. Paged Attention applies the virtual memory insight from operating systems: divide KV storage into fixed-size physical "blocks" (pages), maintain a per-sequence block table that maps logical positions to physical locations, and let a block allocator hand out and reclaim blocks freely as sequences start and end. Combined with Grouped Query Attention, the full system — GQPA — achieves both memory efficiency (paging eliminates fragmentation) and bandwidth efficiency (GQA reduces the number of KV entries to read per query).*

---

## What Is Paged Attention?

Let's start with the problem it solves. In a naive KV cache implementation, you pre-allocate a contiguous buffer of size `max_seq_len × n_heads × d_head` for each sequence in the batch. Two things go wrong:

1. **Fragmentation:** A sequence of 200 tokens wastes nearly all of a 4096-token allocation.
2. **Rigidity:** You can't grow a sequence past its pre-allocated size without copying.

**The paging solution** (introduced in [vLLM](https://arxiv.org/abs/2309.06180)) borrows directly from OS virtual memory:

```
Physical memory = a pool of fixed-size blocks  (e.g., 16 tokens each)
Block table[sequence][logical_slot] → physical_block_id
```

During attention, to look up key/value at logical position $p$ in sequence $b$:
```python
block_id  = block_table[b, p // block_size]   # which physical block?
offset    = p % block_size                     # where inside the block?
k = kv_blocks[block_id, offset, kv_head, :]   # the actual KV vector
```

Sequences grow by requesting new blocks from the pool — no copying. When a sequence ends, its blocks are returned to the pool and immediately reusable. Memory fragmentation drops from potentially 50% to near-zero.

**GQPA adds GQA on top:** after computing `kv_h = query_h // group_size`, every KV lookup goes through the block table for that specific KV head. The two mechanisms compose cleanly.

## Level 0 — Pure Python: The Block Table

Let's build the block table from scratch. We'll use a tiny example: 3 sequences sharing a pool of physical blocks, with a block size of 4 tokens. The goal is to see the `block_table → physical_block → kv_blocks` indirection in the simplest possible form.

In [ ]:
import math

# ── dimensions ──
T, d_head, H, G    = 6, 2, 4, 2    # 6 tokens, head_dim=2, 4 Q-heads, 2 KV-heads
group_size = H // G                  # 2 Q-heads per KV-head
block_size = 2                       # tokens per physical page
n_phys_blocks = 8                    # total pages in the pool

# ── physical KV block storage: [n_phys_blocks, block_size, G, d_head] ──
import random; random.seed(7)
def rnd(n): return [random.gauss(0,.5) for _ in range(n)]

kv_blocks = [
    [[rnd(d_head) for _ in range(G)] for _ in range(block_size)]
    for _ in range(n_phys_blocks)
]

# ── block table for our single sequence ──
# sequence uses logical blocks 0..T//block_size-1
# each logical block points to a physical block (scattered in the pool)
block_table = [3, 1, 6]   # logical block 0→phys 3, block 1→phys 1, block 2→phys 6

print(f"Sequence length: {T} tokens, block_size: {block_size}")
print(f"Logical blocks needed: {math.ceil(T/block_size)}")
print(f"block_table: {block_table}  (logical→physical mapping)")
print()

def get_kv(pos, kv_h):
    """Look up K or V for token at logical position pos, KV head kv_h."""    logical_block = pos // block_size
    block_offset  = pos % block_size
    phys_block    = block_table[logical_block]
    return kv_blocks[phys_block][block_offset][kv_h]

for pos in range(T):
    for g in range(G):
        kv = get_kv(pos, g)
        print(f"  KV[pos={pos}, kv_h={g}] from phys_block={block_table[pos//block_size]}"
              f" offset={pos%block_size}: {[round(v,3) for v in kv]}")


## Verbose Version — Full GQPA Attention

Now let's combine the GQA head routing (`kv_h = h // group_size`) with the paged KV lookup. We'll trace through a complete attention computation for one query token, printing every block table lookup along the way so the indirection chain is impossible to miss.

In [ ]:
# ── query vectors: Q[head][token][dim] ──
Q = [
    [[0.6, 0.3], [0.1, 0.9], [0.4, 0.5], [0.8, 0.2], [0.3, 0.7], [0.5, 0.4]],  # h=0 → kv=0
    [[0.2, 0.7], [0.7, 0.3], [0.5, 0.6], [0.1, 0.8], [0.6, 0.2], [0.4, 0.5]],  # h=1 → kv=0
    [[0.9, 0.1], [0.4, 0.6], [0.3, 0.8], [0.7, 0.4], [0.2, 0.9], [0.8, 0.3]],  # h=2 → kv=1
    [[0.3, 0.5], [0.6, 0.4], [0.8, 0.1], [0.5, 0.7], [0.4, 0.3], [0.7, 0.6]],  # h=3 → kv=1
]

def softmax_1d(scores):
    m=max(scores); exps=[math.exp(s-m) for s in scores]; Z=sum(exps)
    return [e/Z for e in exps]

output = [[[0.0]*d_head for _ in range(T)] for _ in range(H)]

print("=== GQPA forward pass ===")
for h in range(H):
    kv_h = h // group_size
    print(f"\n[Q-head {h}] → kv_head {kv_h}")
    for t in range(T):
        scores = []
        for s in range(T):
            k = get_kv(s, kv_h)   # ← paged gather
            score = sum(Q[h][t][k_dim]*k[k_dim] for k_dim in range(d_head)) / math.sqrt(d_head)
            scores.append(score)
        weights = softmax_1d(scores)
        for dk in range(d_head):
            output[h][t][dk] = sum(weights[s]*get_kv(s, kv_h)[dk] for s in range(T))
        print(f"  t={t}: scores={[round(s,3) for s in scores]}  "
              f"out={[round(v,4) for v in output[h][t]]}")


## The Formula

$$\boxed{O_{h,t} = \text{softmax}\!\left(\frac{Q_{h,t}\; K_{\kappa(h)}^{\text{paged},\top}}{\sqrt{d_k}}\right) V_{\kappa(h)}^{\text{paged}}}$$

where the paged KV lookup is:

$$K_{\kappa(h), s} = \texttt{kv\_blocks}[\texttt{block\_table}[b,\, \lfloor s/B \rfloor],\; s \bmod B,\; \kappa(h),\; :]$$

| Symbol | Meaning |
|--------|---------|
| $\kappa(h) = \lfloor h/g \rfloor$ | GQA: which KV head serves query head $h$ |
| $B$ | Block size (tokens per physical page) |
| $\texttt{block\_table}[b, i]$ | Physical block id for sequence $b$, logical slot $i$ |
| $\texttt{kv\_blocks}[p, o, g, :]$ | KV vector at physical block $p$, offset $o$, KV head $g$ |

> **Say it out loud:** "GQPA for head $h$, token $t$: route to KV head $\kappa(h)$ via GQA, then for each context position $s$, follow the block table to find the physical block, read the KV at the right offset, and run standard softmax attention."

**Practical memory savings from combining GQA + paging:**
- GQA reduces KV head count: $H/G$ fewer KV entries per token
- Paging eliminates wasted allocation: near-zero fragmentation
- Together: 4–8× effective memory reduction vs naïve MHA with contiguous allocation

## CuTe Layout Python Emulation

Let's model the block table as a flat integer array addressed by a 2D Layout, and the physical KV store as a 4D Layout `[num_blocks, block_size, n_kv_heads, d_head]`. The KV lookup becomes a chain of two Layout-indexed reads: first into the block table to get a physical block id, then into the physical KV store at that id.

In [ ]:
class Layout:
    def __init__(self, shape):
        self.shape=tuple(shape); s,running=[],1
        for sz in reversed(shape): s.insert(0,running); running*=sz
        self.stride=tuple(s)
    def __call__(self,*c): return sum(ci*si for ci,si in zip(c,self.stride))

# Flatten block table: shape [num_logical_blocks]
# kv_blocks flat: shape [n_phys_blocks, block_size, G, d_head]
lBT  = Layout([len(block_table)])
lKV  = Layout([n_phys_blocks, block_size, G, d_head])
bt_flat  = block_table[:]
kv_flat  = [v for pb in kv_blocks for row in pb for head_kv in row for v in head_kv]

def paged_gather(pos, kv_h):
    lb = pos // block_size
    offset = pos % block_size
    phys = bt_flat[lBT(lb)]
    return [kv_flat[lKV(phys, offset, kv_h, dk)] for dk in range(d_head)]

def gqpa_cute(Q, H, G, T, d_head):
    group_size = H // G
    out = [[[0.0]*d_head for _ in range(T)] for _ in range(H)]
    for h in range(H):
        kv_h = h // group_size
        for t in range(T):
            scores = [
                sum(Q[h][t][k]*paged_gather(s,kv_h)[k] for k in range(d_head))/math.sqrt(d_head)
                for s in range(T)
            ]
            m=max(scores); exps=[math.exp(s-m) for s in scores]; Z=sum(exps)
            weights=[e/Z for e in exps]
            for dk in range(d_head):
                out[h][t][dk]=sum(weights[s]*paged_gather(s,kv_h)[dk] for s in range(T))
    return out

cute_out = gqpa_cute(Q, H, G, T, d_head)
# verify
tol=1e-9
for h in range(H):
    for t in range(T):
        assert all(abs(cute_out[h][t][k]-output[h][t][k])<tol for k in range(d_head))
print("✓ CuTe Layout GQPA matches naive output")
print("block_table layout:", lBT, " | kv_blocks layout:", lKV)


## CuTeDSL Implementation

In the GPU kernel, the block table lookup is a single pointer indirection: load a 32-bit integer from `block_table[b, s // block_size]`, then use it as the first index into `kv_cache[block_id, s % block_size, kv_h, d]`. This is essentially the GPU equivalent of a virtual memory page walk, except we do it explicitly rather than relying on the MMU.

> **Why one CTA per (head, token, batch)?** Paged attention involves irregular memory access patterns — different sequences may access completely different physical blocks. Assigning each CTA a specific (head, token, batch) triple keeps the work partitioned cleanly, even when access patterns are irregular. Within a CTA, threads parallelize over the `d` head dimension — a perfectly regular strided access.

## Optimization Ladder

| Version | Memory layout | KV gather | Notes |
|---------|--------------|-----------|-------|
| Pure Python | Nested lists | `get_kv(pos, kv_h)` per element | Readable reference; explicit indirection |
| CuTe Layout | Flat array + `Layout` | `lKV(phys, off, kv_h, dk)` | Verifies all index arithmetic |
| `gqpa_kernel` (CuTeDSL) | `kv_cache[phys, off, kv_h, :]` | Single pointer indirection via `block_table` | One CTA per (head, position, batch) |

**A note on block allocators.** In production systems like vLLM, a global block allocator manages the physical block pool: it hands out free blocks when sequences grow and reclaims them when sequences finish. The kernel itself doesn't need to know about allocation — it just reads from whatever block ids the block table says. This clean separation between the "policy" (allocator) and "mechanism" (kernel) is what makes paged attention composable with any scheduling strategy (continuous batching, beam search, speculative decoding, etc.).

## Paging Saves Fragmentation, Not Bandwidth

A common misconception: paged attention is a performance optimization. It is not — at least not directly. The same bytes are read from HBM whether or not they are paged.

**What paging actually saves:**
```
Without paging: pre-allocate max_seq_len per sequence
  Batch of 8 sequences, max_seq_len=4096:
  Reserved: 8 × 4096 × 8 heads × 128 × 2 × 2 (K+V) = 128 MB
  Actually used (avg 1000 tokens): ~32 MB
  Wasted: 75%

With paging (block_size=16):
  Only allocate blocks that are actually needed
  8 × 1000 tokens: exactly 500 physical blocks × 2 heads × ... = 32 MB
  Wasted: ~3% (partial last block per sequence)
```

**Why this matters for RTX 4080 (16 GB VRAM):**
16 GB has to hold: model weights (6–12 GB) + KV cache + activations.
With BF16 weights: ~12 GB for weights → 4 GB for KV cache
With FP8 weights: ~6 GB for weights → 10 GB for KV cache

At 4 GB KV cache, block_size=16:
- Without paging: max batch × max_seq_len is fixed at load time, wasted heavily
- With paging: serve many short and long sequences flexibly, no waste

**The indirect performance benefit:**
Paging enables **larger effective batch sizes** on the same hardware. Larger batches → better SM utilization → higher token throughput per second. The bandwidth saving from GQA + the flexibility from paging are what make long-context LLM serving viable on a single RTX 4080.

**Block table indirection cost:**
The block table lookup (`phys_block = block_table[seq, logical_slot]`) is one extra memory read per KV position per head. At T_ctx=4096: 4096 × 8 heads = 32,768 extra int32 reads = 128 KB. At 380 GB/s: 0.34 µs. Completely negligible.

## Where Paged Attention Fits in the Qwen3-8B Decode Step

Paged attention replaces step ⑧ (KV cache append) and the KV reads in step ⑨ (attention) with block table indirections. The attention math is unchanged; only the memory layout differs.

```
Without paging (naive):
  KV cache: one contiguous [max_seq_len, 8, 128] buffer per sequence, pre-allocated at max size
  Step ⑧:  k_cache[t, kv_h, :] = k_new     ← direct write at logical position t
  Step ⑨:  k = k_cache[0:t, kv_h, :]       ← contiguous slice

With paging (block_size=16):
  Physical storage: pool of [block_size, 8, 128] blocks
  Per-sequence state: block_table[seq_id, logical_block] → physical_block_id

  Step ⑧ (paged write):
    logical_block = t // 16
    offset        = t % 16
    phys          = block_table[seq_id, logical_block]
    kv_blocks[phys, offset, kv_h, :] = k_new

  Step ⑨ (paged read, inside attention kernel):
    for s in 0..t-1:
      phys  = block_table[seq_id, s // 16]
      k_s   = kv_blocks[phys, s % 16, kv_h, :]
      score = q · k_s / sqrt(128)
```

### Memory freed by paging on RTX 4080

```
RTX 4080: 16 GB VRAM
Qwen3-8B BF16 weights: 15.2 GB → barely fits, almost no room for KV cache
Qwen3-8B FP8 weights:   7.6 GB → 8.4 GB free for KV cache

At 8.4 GB KV cache, block_size=16, BF16:
  Block size: 16 × 8 × 128 × 2 × 2 bytes (K+V) = 65,536 bytes = 64 KB
  Blocks available: 8.4 GB / 64 KB ≈ 131,072 blocks
  At avg 1000 tokens/seq → 63 blocks per seq → ~2,080 simultaneous sequences

  Without paging at avg 1000 tokens (max_seq=4096 allocated):
    8.4 GB / (4096 × 8 × 128 × 4 bytes) = 500 allocations max, but ~75% wasted → ~125 usable
  With paging: ~2,080 sequences → 16× improvement in concurrent capacity
```

The paging gain grows as the gap between average and maximum sequence length grows.

## Paged Attention Implementations in the Wild

### vLLM (original)

Paged attention was introduced in the vLLM paper (Kwon et al., 2023).
- `vllm/core/block_manager.py` — block allocator and block table management
- `vllm/csrc/attention/paged_attention_v2.cu` — CUDA kernel that reads K/V through block tables
- Block size is configurable: 16, 32, or 128 tokens. RTX 4080 typically uses 16 or 32.
- Physical block pool is pre-allocated at server startup based on available VRAM

### SGLang

`sglang/srt/mem_pool.py` — similar block allocator with one key addition: **RadixAttention**.
Sequences with shared prefixes (e.g., the same system prompt) share physical blocks via reference counting.
Copy-on-write semantics mean a block is only duplicated when a sequence diverges.
This turns the block table into a radix tree over token sequences.

### TGI (Text Generation Inference, HuggingFace)

Uses paged attention via the FlashInfer backend.
`text-generation-inference/server/text_generation_server/models/flash_causal_lm.py`

### LightLLM

Distinguishes "token attention" (block_size=1) from standard paged attention.
With block_size=1, each token gets its own page — zero intra-block padding waste.
Useful for variable-length batches where block_size=16 would waste half of every last block.

### The block_size tradeoff

| block_size | Internal fragmentation | Block table size | Cache locality |
|------------|----------------------|------------------|----------------|
| 1 token | 0% | Large | Poor |
| 16 tokens | ≤ 93.75% full | Medium | Good |
| 128 tokens | ≤ 99.2% full | Small | Excellent |

For RTX 4080 serving Qwen3-8B, block_size=16 is the standard recommendation:
minimal fragmentation, block table fits in L2, physical blocks are cache-line aligned.

### What paging doesn't fix

Paging eliminates fragmentation. It does NOT:
- Reduce the bandwidth cost of reading all K and V vectors during attention (the bytes still exist)
- Help with compute-bound scenarios (prefill) — it's purely a memory management mechanism
- Eliminate block table overhead (though one int32 per KV position is negligible: ~0.1% of attention cost)

## One-Sentence Takeaway

Paged attention applies the virtual memory insight from operating systems to GPU KV caches: instead of pre-allocating worst-case contiguous buffers that waste 50–75% of memory to fragmentation, a block allocator hands out fixed-size physical pages on demand and a per-sequence block table maps logical positions to scattered physical locations — eliminating fragmentation so a 16 GB RTX 4080 with FP8 weights can serve 4× more concurrent sequences than a naive contiguous cache would allow, with the block table indirection costing less than 0.1% of total attention time because only one int32 is read per KV position per head.

## Community Optimization Resources for Paged GQA on SM89

### Paging vs bandwidth: what each optimization actually saves

| Optimization | What it saves | Measured gain | Implementation |
|---|---|---|---|
| Paged attention (block_size=16) | Memory fragmentation | 4–16× more concurrent sequences | vLLM / Config |
| GQA (group_size=4) | KV cache bandwidth | 4× less KV data at decode | Architecture |
| FP8 KV cache | KV bandwidth | 2× less KV data | Quantization |
| CUDA graphs | Kernel launch overhead | 4.0× decode speedup (RTX 4090) | Config |
| Flash-Decoding | SM utilization at long context | 2–3× throughput at T≥8K | Triton / CUDA |
| Static KV cache + CUDA graphs | Both bandwidth AND launch overhead | Combined best for RTX 4080 | Config |

---

### Qwen3-8B KV cache: bandwidth numbers with 36 layers

GQA with `n_kv_heads=8`, `head_dim=128`, BF16 at decode:

| Context length | KV bytes (K+V, 36 layers, BF16) | Time at 380 GB/s |
|---|---|---|
| T=1,000 | 36 × 8 × 128 × 2 × 2 × 1000 = 144 MB | 0.38 ms |
| T=4,096 | 576 MB | 1.52 ms |
| T=8,192 | 1.15 GB | 3.03 ms |
| T=32,768 | 4.6 GB | 12.1 ms |

For comparison, reading all 36 layers of BF16 weights: 13.7 GB → 36 ms. At short context, weight reads dominate. At T≥8K, KV bandwidth becomes significant.

---

### RTX 4080 paging capacity: FP8 weights unlock the cache

```
RTX 4080: 16 GB VRAM

With BF16 weights (~12 GB):
  Remaining: ~4 GB for KV cache
  At block_size=16, BF16: 4 GB / 64 KB/block = ~65,536 blocks
  At avg 1000 tokens/seq → 63 blocks/seq → ~1,040 concurrent sequences

With FP8 weights (~6 GB):
  Remaining: ~10 GB for KV cache
  → ~2,600 concurrent sequences — 2.5× more throughput
```

**Recommendation for RTX 4080 serving:** quantize weights to FP8/INT8 first (fastest gains), then maximize batch size via paging, then add CUDA graphs.

---

### SGLang RadixAttention: shared KV prefixes via copy-on-write

For Qwen3-8B with a fixed system prompt (e.g., a long instruction block):

```
Without prefix sharing:
  Each request: process system prompt tokens + user tokens
  100 concurrent requests × 500 system prompt tokens × 144 KB/token = 14.4 GB → doesn't fit

With RadixAttention (SGLang):
  System prompt KV computed once → stored in a shared radix tree node
  100 requests share the same 500 system prompt physical blocks
  Each request only allocates its own unique user token blocks
  → 100× reduction in system prompt KV memory usage
```

RadixAttention uses reference counting and copy-on-write semantics: a shared block is only duplicated when a sequence diverges (e.g., different continuations of the same prefix).

**Reference:** `sgl-project/sglang` — `sglang/srt/mem_pool.py`

---

### Block size tradeoff for RTX 4080

| `block_size` | Memory waste | Block table reads | Best for |
|---|---|---|---|
| 1 token | 0% | Many (1 per KV position) | Max flexibility, worst cache locality |
| 16 tokens | ≤6.25% wasted last block | 1 per 16 positions | **Recommended for RTX 4080** |
| 64 tokens | ≤1.6% | 1 per 64 positions | Long-context serving with few sequences |
| 128 tokens | ≤0.8% | 1 per 128 positions | Throughput-focused; large block pool needed |

Standard vLLM default is `block_size=16`. For RTX 4080 + Qwen3-8B, this is correct — physical blocks are cache-line aligned, the block table fits in L2, and fragmentation overhead is minimal.

---

### References

- **vLLM paged attention:** `vllm-project/vllm` — `vllm/csrc/attention/paged_attention_v2.cu` and `vllm/core/block_manager.py`
- **FlashInfer paged decode:** `flashinfer-ai/flashinfer` — `flashinfer.decode.single_decode_with_kv_cache` with block tables + GQA
- **SGLang RadixAttention:** `sgl-project/sglang` — prefix caching via radix tree over KV blocks
- **PagedAttention paper (Kwon et al., 2023):** `arxiv.org/abs/2309.06180` — the original vLLM system paper introducing the block table abstraction